In [1]:
import torch
import pickle
import numpy as np
import pandas as pd
import os
import json 
from os.path import dirname

#RS
from torch.utils.data import WeightedRandomSampler



root_path = dirname(os.getcwd()) + "/SEPH_OUTCOME"

pd.set_option("display.max_columns", None)
data_dir = root_path + "/data/datasets/original/"
data_dir_processed = root_path + "/data/datasets/processed/"
data_dir_graphs = root_path + "/data/datasets/graphs_repair/"

print("CWD:", os.getcwd())
print(root_path, data_dir, data_dir_processed, data_dir_graphs, sep="\n")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# device = "cpu"

CWD: /home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_OUTCOME
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_OUTCOME
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_OUTCOME/data/datasets/original/
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_OUTCOME/data/datasets/processed/
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_OUTCOME/data/datasets/graphs_repair/


In [2]:
with open("data/dataset_features.json", 'r') as file:
    datasets_info = json.load(file)

In [3]:
list(datasets_info.keys())

['BPIC11_f1', 'sepsis_cases_1', 'sepsis_cases_4', 'BPIC15_common']

In [4]:
dataset = "BPIC11_f1" #decide which dataset to work on

In [5]:
#if dataset.startswith("BPIC15"):
#    with open("data/dataset_features.json", 'r') as file:
#        dataset_info = json.load(file)["BPIC15_common"]
#else:
#    with open("data/dataset_features.json", 'r') as file:
#        dataset_info = json.load(file)[dataset]


with open("data/dataset_features.json", 'r') as file:
        dataset_info = json.load(file)[dataset]

In [6]:
categorical_columns = dataset_info["categorical"]
real_value_columns = dataset_info["numerical"]

In [7]:
tab_all = pd.read_csv(data_dir_processed+dataset+"_processed_all.csv")
tab_all.head()

,Diagnosis,Treatment code,Diagnosis code,Specialism code,Diagnosis Treatment Combination ID,Age,CaseID,Label,Activity,Producer code,Section,Specialism code.1,group,Number of executions,time:timestamp,timesincemidnight,month,weekday,hour,timesincelastevent,timesincecasestart,event_nr,open_cases
0,maligniteit cervix,TC103,M13,SC61,DTC376907,33,0,regular,AC410100,SRTH,Section 5,SC61,Radiotherapy,1,1.104692e+09,1380,1,6,23,0.0,0.0,1,5
1,maligniteit cervix,TC103,M13,SC61,DTC376907,33,0,regular,AC419100,SRTH,Section 5,SC61,Radiotherapy,1,1.104692e+09,1380,1,6,23,0.0,0.0,2,5
2,maligniteit cervix,TC103,M13,SC61,DTC376907,33,0,regular,AC10107,SGEH,Section 2,SC7,Nursing ward,1,1.104865e+09,1380,1,1,23,0.0,2880.0,3,5
3,maligniteit cervix,TC103,M13,SC61,DTC376907,33,0,regular,339486E,SGEC,Section 2,SC7,Obstetrics & Gynaecology clinic,1,1.104865e+09,1380,1,1,23,0.0,2880.0,4,5
4,maligniteit cervix,TC103,M13,SC61,DTC376907,33,0,regular,AC410100,SGEH,Section 2,SC7,Nursing ward,1,1.104865e+09,1380,1,1,23,0.0,2880.0,5,5


In [8]:
import random

torch.manual_seed(0)
torch.cuda.manual_seed(0)
random.seed(0)
np.random.seed(0)

In [9]:
with open(data_dir_graphs + dataset + "_TRAIN_repair.pkl", "rb") as f:
    X_train = pickle.load(f)
with open(data_dir_graphs + dataset + "_VALID_repair.pkl", "rb") as f:
    X_valid = pickle.load(f)
with open(data_dir_graphs + dataset + "_TEST_repair.pkl", "rb") as f:
    X_test = pickle.load(f)

In [10]:

from torch_geometric.data import Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.transforms import ToUndirected, NormalizeFeatures

transform = ToUndirected()

with torch.no_grad():
        for i in range(len(X_train)):
                X_train[i] = transform(X_train[i])
        for i in range(len(X_valid)):
                X_valid[i] = transform(X_valid[i])
        for i in range(len(X_test)):
                X_test[i] = transform(X_test[i])
    


In [11]:
edge_types = set()
node_types = set()
for i in range(len(X_train)):
    n, edge_type = X_train[i].metadata()
    for x in n:
        node_types.add(x)
    for x in edge_type:
        edge_types.add(x)
for i in range(len(X_valid)):
    n, edge_type = X_valid[i].metadata()
    for x in n:
        node_types.add(x)
    for x in edge_type:
        edge_types.add(x)
for i in range(len(X_test)):
    n, edge_type = X_test[i].metadata()
    for x in n:
        node_types.add(x)
    for x in edge_type:
        edge_types.add(x)



In [12]:
node_types = list(node_types)
edge_types = list(edge_types)

In [13]:
node_types

['Diagnosis Treatment Combination ID',
 'timesincelastevent',
 'Number of executions',
 'Specialism code.1',
 'Activity',
 'Diagnosis code',
 'Producer code',
 'timesincemidnight',
 'Specialism code',
 'hour',
 'open_cases',
 'timesincecasestart',
 'weekday',
 'Section',
 'Treatment code',
 'Diagnosis',
 'group',
 'month',
 'event_nr',
 'Age']

In [14]:
edge_types

[('open_cases', 'rev_related_to', 'Activity'),
 ('Activity', 'related_to', 'Section'),
 ('Specialism code', 'rev_related_to', 'Activity'),
 ('month', 'rev_related_to', 'Activity'),
 ('Number of executions', 'related_to', 'Number of executions'),
 ('Diagnosis code', 'rev_related_to', 'Activity'),
 ('Activity', 'related_to', 'Diagnosis code'),
 ('Activity', 'related_to', 'timesincemidnight'),
 ('Producer code', 'related_to', 'Producer code'),
 ('weekday', 'related_to', 'weekday'),
 ('Specialism code.1', 'related_to', 'Specialism code.1'),
 ('Treatment code', 'rev_related_to', 'Activity'),
 ('timesincemidnight', 'related_to', 'timesincemidnight'),
 ('Diagnosis Treatment Combination ID', 'rev_related_to', 'Activity'),
 ('Activity', 'related_to', 'Producer code'),
 ('Activity', 'related_to', 'Specialism code'),
 ('Activity', 'followed_by', 'Activity'),
 ('Activity', 'related_to', 'open_cases'),
 ('timesincecasestart', 'rev_related_to', 'Activity'),
 ('Treatment code', 'related_to', 'Treatme

## Hyperopt

In [15]:
print(f"PyTorch: {torch.__version__}")
#print(f"TorchVision: {torchvision.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

PyTorch: 2.6.0+cu124
CUDA Available: False


In [16]:
from ax.service.managed_loop import optimize

In [17]:
from torch_geometric.nn import (
    HeteroConv,
    global_mean_pool,
    GATv2Conv,
    SAGEConv,
    TransformerConv
)
from torch.nn import (
    ModuleList,
    Module,
    Linear
  )
from typing_extensions import Self

In [18]:
from torch_geometric.nn import HeteroConv, global_mean_pool, SAGEConv
from torch.nn import Module, ModuleList, Sequential, Linear, Dropout, BatchNorm1d, ReLU
import torch.nn.functional as F

class HGNN(Module):
    def __init__(self, nodes_relations, parameters):
        super().__init__()
        hid           = parameters["hid"]
        layers        = parameters["layers"]
        aggregation   = parameters["aggregation"]
        dropout_p     = parameters.get("dropout", 0.1)

        # 1) stack of hetero‐message‐passing layers
        self.convs = ModuleList()
        self.bns   = ModuleList()
        self.dps   = ModuleList()
        for _ in range(layers):
            # hetero‐conv over each relation
            conv = HeteroConv(
                { rel: SAGEConv((-1, -1), aggr=aggregation, out_channels=hid, normalize=False)
                  for rel in nodes_relations },
                aggr=aggregation,
            )
            self.convs.append(conv)
            # batchnorm + dropout for the hidden dim
            self.bns.append(BatchNorm1d(hid))
            self.dps.append(Dropout(dropout_p))

        # 2) final graph‐classification head: MLP hid→hid→1
        self.classifier = Sequential(
            Linear(hid, hid),
            ReLU(),
            BatchNorm1d(hid),
            Dropout(dropout_p),
            Linear(hid, 1),
        )

    def forward(self, batch):
        x_dict    = batch.x_dict
        edge_dict = batch.edge_index_dict

        # --- message‑passing with BN/ReLU/Dropout after each conv ---
        for conv, bn, dp in zip(self.convs, self.bns, self.dps):
            x_dict = conv(x_dict, edge_dict)

            # normalize + activate + drop only on the “Activity” embeddings
            act = x_dict["Activity"]
            act = bn(act)
            act = F.relu(act)
            act = dp(act)
            x_dict["Activity"] = act

            # for all other node types, just ReLU
            for nt, x in x_dict.items():
                if nt != "Activity":
                    x_dict[nt] = F.relu(x)

        # --- graph‑level readout on “Activity” nodes ---
        h_act  = x_dict["Activity"]
        pooled = global_mean_pool(h_act, batch["Activity"].batch)

        # --- final MLP head → logits ---
        logits = self.classifier(pooled).view(-1)
        return logits



    

In [19]:
from torcheval.metrics.functional import multiclass_accuracy, multiclass_f1_score
import torch.nn as nn
import time

In [20]:
#Weighted Random Sampling
#Pull out all labels into a single 1D tensor of 0/1
y_train = torch.cat([g.y for g in X_train]).long()
#count examples per class
class_counts = torch.bincount(y_train)
#Inverse frequency
class_weights = 1.0 / class_counts.float()

print("class_counts:", class_counts.tolist())
print("class_weights:", class_weights.tolist())


# number of negatives & positives
n_neg, n_pos = class_counts.tolist()

# the weight for positive class = n_neg / n_pos
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float, device=device)
print("Using BCEWithLogitsLoss pos_weight =", pos_weight.item())

#Assign each sample the weight of it's class
sample_weights = class_weights[y_train]
#create a sampler that draws 'len(sample_weights)' samples per epoch
sampler = WeightedRandomSampler(
     weights=sample_weights,
     num_samples=len(sample_weights),
     replacement=True,
 )

class_counts: [320, 197]
class_weights: [0.0031250000465661287, 0.005076142027974129]
Using BCEWithLogitsLoss pos_weight = 1.6243654489517212


In [21]:
from collections import Counter

# Draw 10,000 “indices” from the sampler
sampled_indices = list(WeightedRandomSampler(
    weights=sample_weights,
    num_samples=500,
    replacement=True
))

# Map each index back to its label
sampled_labels = [ y_train[idx].item() for idx in sampled_indices ]
print(Counter(sampled_labels))

Counter({1: 261, 0: 239})


In [22]:
from copy import deepcopy
from tqdm.notebook import tqdm

def train_hgnn(config, epochs=20):
    
    print(config)

    net = HGNN(
        parameters=config,
        nodes_relations=edge_types,
    )
    net = net.to(device)

    # loss for graph binary classification
    #loss_fn = nn.BCEWithLogitsLoss()
    loss_fn = nn.BCEWithLogitsLoss()

    #train_loader = DataLoader(X_train, batch_size=config["batch_size"], shuffle=True)
    train_loader = DataLoader(
        X_train,
        batch_size=config["batch_size"],
        sampler=sampler,     # ← use the balanced sampler
        shuffle=False,       # ← don’t shuffle when using sampler
    )


    valid_loader = DataLoader(X_valid, batch_size=config["batch_size"], shuffle=False)


    optimizer = torch.optim.Adam(net.parameters(), lr=config["lr"])

    best_model = None
    best_loss = float("inf")
    patience = 5
    pat_count = 0

    torch.cuda.empty_cache()

    for epoch in tqdm(range(0, epochs)):
        start_time = time.time()

        #print(f"Epoch: {epoch}\n")

        net.train()
        for _, x in enumerate(train_loader):
            x = x.to(device)

            optimizer.zero_grad()       

            logits = net(x) #shape [batch_size]
            labels = x.y.float() #shape [batch_size]
            loss = loss_fn(logits, labels)

            loss.backward()
            optimizer.step()

        #--validation--
        #running_loss = 0.0
        #correct = 0
        #total = 0
        running_loss = 0.0
        all_logits = []
        all_labels = []

        net.eval()
        with torch.no_grad():
            for x in valid_loader:
                x = x.to(device)
                logits = net(x)
                labels = x.y 

                running_loss +=loss_fn(logits, labels.float()).item()

                #compute binary predictions
                #preds = (torch.sigmoid(logits) > 0.5).long()
                #correct += (preds == labels).sum().item()
                #total += labels.size(0)

                #accumulate for AUC
                all_logits.append(logits.cpu())
                all_labels.append(labels.cpu())


        val_loss = running_loss / len(valid_loader)
        #val_acc = correct / total

        #Concatenate and compute AUC
        all_logits = torch.cat(all_logits)
        all_labels = torch.cat(all_labels).numpy()
        all_probs  = torch.sigmoid(all_logits).numpy()
        from sklearn.metrics import roc_auc_score
        val_auc = roc_auc_score(all_labels, all_probs)


        # Early stopping 
        if val_loss < best_loss:
            best_loss  = val_loss
            best_model = deepcopy(net)
            pat_count  = 0
        else:
            pat_count += 1
            if pat_count >= patience:
                break

    return best_model

In [23]:
from torch_geometric.loader import DataLoader
import torch.nn as nn
import torch

def test_hgnn(net):
    """
    Evaluate a trained HGNN (graph‑level classifier) on X_test.
    Returns a dict with test loss and accuracy.
    """
    test_loader = DataLoader(X_test, batch_size=128, shuffle=False)
    loss_fn = nn.BCEWithLogitsLoss()

    net.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    all_logits = []
    all_labels = []

    with torch.no_grad():
        for x in test_loader:
            x = x.to(device)
            logits = net(x)           
            labels = x.y              

            # accumulate loss
            total_loss += loss_fn(logits, labels.float()).item()

            # binary predictions & accuracy
            #preds = (torch.sigmoid(logits) > 0.5).long()
            #correct += (preds == labels).sum().item()
            #total += labels.size(0)

            # accumulate for AUC
            all_logits.append(logits.cpu())
            all_labels.append(labels.cpu())


    avg_loss = total_loss / len(test_loader)
    #accuracy = correct / total
    # compute AUC
    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels).numpy()
    all_probs  = torch.sigmoid(all_logits).numpy()
    from sklearn.metrics import roc_auc_score
    test_auc = roc_auc_score(all_labels, all_probs)


    #print(f"Test loss: {avg_loss:.4f}, Test accuracy: {accuracy:.4f}")
    #return {"test_loss": avg_loss, "test_acc": accuracy}
    print(f"Test loss: {avg_loss:.4f}, Test ROC‑AUC: {test_auc:.4f}")
    return {"test_loss": avg_loss, "test_auc": test_auc}


In [24]:
# Calculate unique counts for categorical columns
list_unique = {col: len(tab_all[col].unique()) for col in categorical_columns}

#outputcat = {k : len(list_unique[k]) for k in list_unique}
outputcat = list_unique
outputreal = real_value_columns
print(outputcat)
print(outputreal)

{'Diagnosis': 105, 'Treatment code': 43, 'Diagnosis code': 11, 'Specialism code': 3, 'Diagnosis Treatment Combination ID': 799, 'CaseID': 1140, 'Activity': 193, 'Producer code': 52, 'Section': 7, 'Specialism code.1': 14, 'group': 24}
['Age', 'Number of executions', 'timesincemidnight', 'month', 'weekday', 'hour', 'timesincelastevent', 'timesincecasestart', 'event_nr', 'open_cases']


In [25]:
def train_evaluate(config):
    trained_net = train_hgnn(config, epochs=50)
    return test_hgnn(trained_net)

In [26]:
import logging

logging.getLogger("root").setLevel(logging.ERROR)

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [27]:
y_train = torch.cat([batch.y for batch in X_train]).float()
num_true = y_train.sum().item()
num_false = len(y_train) - num_true

# Assign weights to BCEWithLogitsLoss
pos_weight = num_false / num_true
pos_weight = torch.tensor([num_false / num_true], device=device)

print("pos_weight: ", pos_weight)

pos_weight:  tensor([1.6244])


In [28]:
sample_config = {
    "hid":          128, #128
    "layers":       2, #2
    "lr":           1e-3,
    "batch_size":   256, #256
    "aggregation": "mean",
}


# 1-epoch train just to exercise the code-path and see prints
net = train_hgnn(sample_config, epochs=1)

# run the test (with your debug prints enabled)
res = test_hgnn(net)
print("test_hgnn returned:", res)


{'hid': 128, 'layers': 2, 'lr': 0.001, 'batch_size': 256, 'aggregation': 'mean'}


  0%|          | 0/1 [00:00<?, ?it/s]

Test loss: 0.6800, Test ROC‑AUC: 0.5041
test_hgnn returned: {'test_loss': 0.680025060971578, 'test_auc': 0.5041118421052632}


In [29]:
best_parameters, values, experiment, model = optimize(
    parameters=[
        {"name": "hid", "type": "choice", "values": [64,128,256,512], "value_type": "int", "is_ordered" : True,"sort_values":False},
        #{"name": "hid", "type": "choice", "values": [512], "value_type": "int", "is_ordered" : True,"sort_values":False},
        {"name": "layers", "type": "choice", "values": [2, 3, 4, 5], "value_type": "int", "is_ordered" : True, "sort_values":False},
        #{"name": "layers", "type": "choice", "values": [2], "value_type": "int", "is_ordered" : True, "sort_values":False},
        {"name": "lr", "type": "range", "bounds": [1e-4, 1e-1], "value_type": "float", "log_scale": True},
        {"name": "batch_size", "type": "choice", "values": [128,256,512], "value_type": "int", "is_ordered" : True,"sort_values":False}, 
        
        #{"name": "heads", "type": "choice", "values": [1,2], "value_type": "int", "is_ordered" : True,"sort_values":False},
        #{"name": "heads", "type": "choice", "values": [1], "value_type": "int", "is_ordered" : True,"sort_values":False},
        
        {"name": "aggregation", "type" : "choice", "values" :["sum", "mean", "max"], "value_type" : "str"}
        #{"name": "aggregation", "type" : "choice", "values" :["max"], "value_type" : "str"},
     
    ],
  
    evaluation_function=train_evaluate,
    objective_name='test_auc',
    arms_per_trial=1,
    minimize = False,
    random_seed = 123,
    total_trials = 30
)

print(best_parameters)
means, covariances = values
print(means)
print(experiment)

/home/matteo/Documents/GNN-test2/SEPH_MODELS/env3/lib/python3.10/site-packages/ax/service/utils/instantiation.py:248: AxParameterWarning: `is_ordered` is not specified for `ChoiceParameter` "aggregation". Defaulting to `False`  since the parameter is a string with more than 2 choices.. To override this behavior (or avoid this warning), specify `is_ordered` during `ChoiceParameter` construction. Note that choice parameters with exactly 2 choices are always considered ordered and that the user-supplied `is_ordered` has no effect in this particular case.
  return ChoiceParameter(
/home/matteo/Documents/GNN-test2/SEPH_MODELS/env3/lib/python3.10/site-packages/ax/service/utils/instantiation.py:248: AxParameterWarning: `sort_values` is not specified for `ChoiceParameter` "aggregation". Defaulting to `False` for parameters of `ParameterType` STRING. To override this behavior (or avoid this warning), specify `sort_values` during `ChoiceParameter` construction.
  return ChoiceParameter(
[INFO 06

{'hid': 64, 'layers': 4, 'lr': 0.001671979996274695, 'batch_size': 512, 'aggregation': 'sum'}


  0%|          | 0/50 [00:00<?, ?it/s]

[INFO 06-20 02:36:27] ax.core.experiment: Attached data has some metrics ({'test_loss'}) that are not among the metrics on this experiment. Note that attaching data will not automatically add those metrics to the experiment. For these metrics to be automatically fetched by `experiment.fetch_data`, add them via `experiment.add_tracking_metric` or update the experiment's optimization config.
[INFO 06-20 02:36:27] ax.service.managed_loop: Running optimization trial 2...
[ERROR 06-20 02:36:27] ax.core.observation: Data contains metric test_loss that has not been added to the experiment. You can either update the `optimization_config` or attach it as a tracking metric using `Experiment.add_tracking_metrics` or `AxClient.add_tracking_metrics`. Ignoring all data for metric test_loss.
NoneType: None


Test loss: 0.6803, Test ROC‑AUC: 0.5075
{'hid': 256, 'layers': 2, 'lr': 0.027575328715146064, 'batch_size': 128, 'aggregation': 'max'}


...

  0%|          | 0/50 [00:00<?, ?it/s]

[INFO 06-20 02:52:09] ax.core.experiment: Attached data has some metrics ({'test_loss'}) that are not among the metrics on this experiment. Note that attaching data will not automatically add those metrics to the experiment. For these metrics to be automatically fetched by `experiment.fetch_data`, add them via `experiment.add_tracking_metric` or update the experiment's optimization config.
[ERROR 06-20 02:52:09] ax.core.observation: Data contains metric test_loss that has not been added to the experiment. You can either update the `optimization_config` or attach it as a tracking metric using `Experiment.add_tracking_metrics` or `AxClient.add_tracking_metrics`. Ignoring all data for metric test_loss.
NoneType: None
[ERROR 06-20 02:52:09] ax.core.observation: Data contains metric test_loss that has not been added to the experiment. You can either update the `optimization_config` or attach it as a tracking metric using `Experiment.add_tracking_metrics` or `AxClient.add_tracking_metrics`. 

Test loss: 29.1299, Test ROC‑AUC: 0.4772


[WARNING 06-20 02:52:10] ax.modelbridge.cross_validation: Metric test_auc was unable to be reliably fit.
[WARNING 06-20 02:52:10] ax.service.utils.best_point: Model fit is poor; falling back on raw data for best point.
[WARNING 06-20 02:52:10] ax.service.utils.best_point: Model fit is poor and data on objective metric test_auc is noisy; interpret best points results carefully.


{'hid': 64, 'layers': 2, 'lr': 0.00030100727600525814, 'batch_size': 512, 'aggregation': 'sum'}
{'test_loss': 0.6816311478614807, 'test_auc': 0.6233735380116958}
Experiment(None)


In [30]:
from ax.service.utils.report_utils import exp_to_df

results = exp_to_df(experiment)

[WARNING 06-20 02:52:10] ax.service.utils.report_utils: Column reason missing for all trials. Not appending column.


In [31]:
results.sort_values(by="test_auc")

,trial_index,arm_name,trial_status,generation_method,test_auc,test_loss,hid,layers,lr,batch_size,aggregation
26,26,26_0,COMPLETED,BoTorch,0.460417,0.681556,64,4,0.001806,256,max
25,25,25_0,COMPLETED,BoTorch,0.462262,0.697866,128,3,0.001468,512,max
17,17,17_0,COMPLETED,BoTorch,0.464711,5.487844,512,2,0.036016,128,max
23,23,23_0,COMPLETED,BoTorch,0.466118,1.240505,128,3,0.002209,512,max
15,15,15_0,COMPLETED,BoTorch,0.467032,0.678335,64,2,0.024918,512,sum
29,29,29_0,COMPLETED,BoTorch,0.477193,29.129939,64,2,0.001802,256,max
6,6,6_0,COMPLETED,Sobol,0.482292,0.921869,256,4,0.004998,128,max
2,2,2_0,COMPLETED,Sobol,0.488999,0.808174,512,5,0.000129,512,max
16,16,16_0,COMPLETED,BoTorch,0.498849,0.679544,64,3,0.000327,512,sum
24,24,24_0,COMPLETED,BoTorch,0.500000,961.203064,512,3,0.027322,128,max


In [32]:
results = results.sort_values(by="test_auc")

In [33]:
results.to_csv(f"results/{dataset}.csv", sep=",")